# Notebook 7 – Backpropagation & Gradients

In this notebook, we will follow how a tiny neural network learns: make a prediction, measure its error, find which way to change its weights, and update them.

## 1. What is a gradient?

A **gradient** tells us how much the loss changes when a model value, such as a weight, changes. For one weight, think of it as the slope of the loss. A positive gradient means increasing the weight makes the loss go up; a negative gradient means increasing it can make the loss go down.

## 2. Why are gradients required?

A neural network may have many weights. Gradients tell us how each weight contributed to the error and which direction to adjust it. Without gradients, we would be changing weights without knowing whether those changes help the model learn.

## 4. Computational graph

A **computational graph** is a simple map of calculations. Our tiny model follows this path:

`w, x, b → prediction (w × x + b) → loss ((prediction − y)²)`

Each arrow shows how one calculation uses the result of an earlier calculation.

## 5. Forward pass

The **forward pass** follows the graph from input to output. With x = 2, w = 1, and b = 0, the prediction is `1 × 2 + 0 = 2`.

## 6. Loss calculation

A **loss** is one number that measures how far a prediction is from the correct answer. If the correct answer y is 5, squared error gives `(2 − 5)² = 9`. During learning, we want to make this number smaller.

## 8. Chain rule and manual gradient

The **chain rule** lets us find how an early value affects the final loss by multiplying the gradients along the path. Here:
- Change in loss for a change in prediction: `2 × (prediction − y) = −6`.
- Change in prediction for a change in w: `x = 2`, so the weight gradient is `(−6) × 2 = −12`.
- Change in prediction for a change in b: `1`, so the bias gradient is `(−6) × 1 = −6`.

The negative signs tell us that increasing w and b should lower this loss. We will let PyTorch calculate these same gradients in a later section.

In [16]:
x = 2.0
y = 5.0
w = 1.0
b = 0.0
learning_rate = 0.05
prediction = w * x + b
loss = (prediction - y) ** 2
gradient_w = 2 * (prediction - y) * x
gradient_b = 2 * (prediction - y)
w_new = w - learning_rate * gradient_w
b_new = b - learning_rate * gradient_b
print("Prediction:", prediction)
print("Loss before update:", loss)
print("Gradient for w:", gradient_w)
print("Gradient for b:", gradient_b)
print("Updated w and b:", w_new, b_new)
print("Loss after update:", (w_new * x + b_new - y) ** 2)

Prediction: 2.0
Loss before update: 9.0
Gradient for w: -12.0
Gradient for b: -6.0
Updated w and b: 1.6 0.30000000000000004
Loss after update: 2.25


## 3. Gradient descent

**Gradient descent** is a way to reduce the loss by repeatedly nudging weights in the direction suggested by their gradients. We take a small step, check the loss again, and repeat. The step size is controlled by the learning rate.

## 7. Backward pass

The **backward pass** starts at the loss and moves backwards through the computational graph. It works out how much each earlier value, including each weight, contributed to the loss. This uses the chain rule explained above.

## 9. Backpropagation

**Backpropagation** is the process of sending error information backwards through a neural network to calculate gradients for its weights. In our tiny example, it sends the loss information back through the prediction to find the gradients for w and b. PyTorch can do these calculations automatically with autograd.

## 4. Let PyTorch calculate the gradient

PyTorch **autograd** records the tensor operations used in the forward pass and builds a computational graph. Set `requires_grad=True` on each value you want gradients for, calculate the loss, then call `.backward()`. PyTorch follows the chain rule and stores each result in that tensor's `.grad` property.

Here we ask for gradients of both the weight and bias. The expected values are the same as our manual calculation: −12 for w and −6 for b.

In [17]:
import torch

x = torch.tensor(2.0)
y = torch.tensor(5.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

prediction = w * x + b       # forward pass
loss = (prediction - y) ** 2  # calculate loss
loss.backward()               # backward pass

print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("Gradient for w:", w.grad.item())
print("Gradient for b:", b.grad.item())

Prediction: 2.0
Loss: 9.0
Gradient for w: -12.0
Gradient for b: -6.0


The gradients should be **−12** for w and **−6** for b, matching our hand calculation. Autograd did the chain-rule work for us. For a model with many layers and parameters, this saves us from deriving every gradient by hand.

## 10. Weight update

After calculating gradients, we update each weight using `new weight = old weight − learning rate × gradient`. For our example, the learning rate is 0.05, so `w = 1 − 0.05 × (−12) = 1.6` and `b = 0 − 0.05 × (−6) = 0.3`. The prediction becomes 3.5 and the loss drops from 9 to 2.25. This is one gradient-descent update. In training, we reset gradients before the next backward pass so they do not accumulate by accident.

In [18]:
learning_rate = 0.05

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

print("Updated w and b:", w.item(), b.item())
print("Updated prediction:", (w * x + b).item())
print("Updated loss:", ((w * x + b - y) ** 2).item())

Updated w and b: 1.600000023841858 0.30000001192092896
Updated prediction: 3.5
Updated loss: 2.25


The complete learning loop is:

**Forward Pass → Calculate Loss → Backward Pass → Calculate Gradients → Update Weights**

A real training loop repeats these steps over many examples. Next, we will look at optimizers that manage the weight updates for us.